In [3]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

def master_engineering(df):
    # 1. Datas e Horas (Simplificado para evitar ruído)
    df['record_date'] = pd.to_datetime(df['record_date'], format='mixed')
    df['hour'] = df['record_date'].dt.hour
    df['day_week'] = df['record_date'].dt.dayofweek
    
    # 2. Métrica de "Engarrafamento" (Física Pura)
    # Em vez de médias, usamos a proporção de atraso face ao tempo normal
    df['delay_factor'] = df['AVERAGE_TIME_DIFF'] / (df['AVERAGE_FREE_FLOW_TIME'] + 1)
    
    # Velocidade Efetiva vs Velocidade Teórica
    # Distância aproximada = speed * time
    dist = df['AVERAGE_FREE_FLOW_SPEED'] * df['AVERAGE_FREE_FLOW_TIME']
    df['real_speed'] = dist / (df['AVERAGE_FREE_FLOW_TIME'] + df['AVERAGE_TIME_DIFF'] + 0.1)
    
    # Diferença de velocidade (o quão longe estamos do ideal)
    df['speed_loss'] = df['AVERAGE_FREE_FLOW_SPEED'] - df['real_speed']
    
    # 3. Categorização de Vias
    # Ruas lentas (cidade) vs Ruas rápidas (VCI/Autoestrada) reagem diferente ao trânsito
    df['is_highway'] = (df['AVERAGE_FREE_FLOW_SPEED'] > 70).astype(int)
    
    # 4. Clima Binário (Reduzir 10 categorias para 2)
    # O que importa é se está "mau tempo" ou não
    bad_weather = ['Chuva', 'Tempestade', 'Trovoada', 'Chuvoso']
    df['bad_weather'] = df['AVERAGE_CLOUDINESS'].fillna('').apply(
        lambda x: 1 if any(w in str(x) for w in bad_weather) else 0
    )
    
    # 5. Ciclos (Seno/Cosseno)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/23.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/23.0)

    # Colunas a manter (apenas o essencial)
    features = [
        'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_FREE_FLOW_TIME', 'AVERAGE_TIME_DIFF',
        'hour', 'day_week', 'delay_factor', 'real_speed', 'speed_loss', 
        'is_highway', 'bad_weather', 'hour_sin', 'hour_cos'
    ]
    return df[features]

# --- LOAD DATA ---
train = pd.read_csv("training_data.csv", encoding="latin-1")
test = pd.read_csv("test_data.csv", encoding="latin-1")

# --- TARGET MAPPING (ORDINAL) ---
# Muito importante: Low está mais perto de Medium do que de Very_High
target_map = {
    'None': 0,
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Very_High': 4
}
inv_target_map = {v: k for k, v in target_map.items()}

train['AVERAGE_SPEED_DIFF'] = train['AVERAGE_SPEED_DIFF'].fillna('None')
y = train['AVERAGE_SPEED_DIFF'].map(target_map)

X = master_engineering(train)
X_test = master_engineering(test)

# --- MODELAGEM COM STACKING (A ARMA SECRETA) ---
# O Stacking usa os 3 modelos e depois um 4º (LogisticRegression) para decidir o voto

base_models = [
    ('lgb', LGBMClassifier(n_estimators=1000, learning_rate=0.02, num_leaves=31, random_state=42, verbosity=-1)),
    ('xgb', XGBClassifier(n_estimators=1000, learning_rate=0.02, max_depth=6, random_state=42)),
    ('cat', CatBoostClassifier(iterations=1000, learning_rate=0.02, depth=6, random_state=42, verbose=0))
]

# O meta-modelo (final_estimator) aprende a combinar os resultados
stacker = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

print("🚀 Treinando Stacking Ensemble (LGBM + XGB + CatBoost)...")
stacker.fit(X, y)

# --- SUBMISSÃO ---
preds = stacker.predict(X_test)
labels = [inv_target_map[p] for p in preds]

submission = pd.DataFrame({
    "RowId": np.arange(1, len(test) + 1),
    "Speed_Diff": [str(p).replace('_', ' ').title().replace(' ', '_') for p in labels]
})
submission['Speed_Diff'] = submission['Speed_Diff'].replace('None', 'None')

submission.to_csv("submission_master_v7.csv", index=False)
print("Ficheiro v7 gerado com sucesso!")

🚀 Treinando Stacking Ensemble (LGBM + XGB + CatBoost)...
Ficheiro v7 gerado com sucesso!
